In [1]:
# Project 4: Diabetes Population Health Management
# Notebook 5: Complication Prediction Among Diabetics
# Objective: Identify high-risk diabetics for targeted intervention

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
import json
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

print("Environment loaded successfully")

# Load processed data
df = pd.read_csv('../data/processed/brfss_2023_diabetes.csv')

print(f"Loaded: {len(df):,} respondents")

# Filter to diabetics only
diabetics = df[df['has_diabetes'] == 1].copy()

print(f"\nDiabetics subset: {len(diabetics):,} individuals")
print(f"Percentage of total: {len(diabetics)/len(df)*100:.1f}%")

# Create composite complication outcome
# Any cardiovascular event, kidney disease, or vision problems

diabetics['cvd_complication'] = diabetics['CVDINFR4'].apply(
    lambda x: 1 if x == 1 else (0 if x == 2 else np.nan)
)

diabetics['stroke_complication'] = diabetics['CVDSTRK3'].apply(
    lambda x: 1 if x == 1 else (0 if x == 2 else np.nan)
)

diabetics['kidney_complication'] = diabetics['CHCKDNY2'].apply(
    lambda x: 1 if x == 1 else (0 if x == 2 else np.nan)
)

# Composite: any complication
diabetics['any_complication'] = (
    (diabetics['cvd_complication'] == 1) |
    (diabetics['stroke_complication'] == 1) |
    (diabetics['kidney_complication'] == 1)
).astype(int)

print("\nComplication prevalence among diabetics:")
print(f"  Cardiovascular: {diabetics['cvd_complication'].mean():.1%}")
print(f"  Stroke: {diabetics['stroke_complication'].mean():.1%}")
print(f"  Kidney disease: {diabetics['kidney_complication'].mean():.1%}")
print(f"  Any complication: {diabetics['any_complication'].mean():.1%}")

# Create risk prediction features
print("\nCreating risk prediction features...")

# Age when diagnosed
diabetics['diagnosis_age'] = diabetics['DIABAGE3']

# Current age group
diabetics['age_group_num'] = diabetics['_AGE_G']

# Disease duration proxy (current age group - diagnosis age)
# Note: This is approximate with grouped data
diabetics['disease_duration_proxy'] = diabetics['age_group_num'] * 10 - diabetics['diagnosis_age']
diabetics.loc[diabetics['disease_duration_proxy'] < 0, 'disease_duration_proxy'] = 0

# BMI
diabetics['bmi'] = diabetics['bmi_numeric']

# Physical activity
diabetics['physically_active'] = diabetics['EXERANY2'].apply(
    lambda x: 1 if x == 1 else (0 if x == 2 else np.nan)
)

# Sex
diabetics['is_male'] = diabetics['BIRTHSEX'].apply(
    lambda x: 1 if x == 1 else (0 if x == 2 else np.nan)
)

# Eye exam compliance (proxy for disease management)
diabetics['had_eye_exam'] = diabetics['DIABEYE'].apply(
    lambda x: 1 if x == 1 else (0 if x == 2 else np.nan)
)

print("Features created successfully")

# Define feature list
features = [
    'diagnosis_age',
    'age_group_num',
    'disease_duration_proxy',
    'bmi',
    'physically_active',
    'is_male',
    'had_eye_exam'
]

# Create modeling dataset
model_vars = features + ['any_complication', '_LLCPWT']
df_model = diabetics[model_vars].dropna()

print(f"\nModeling dataset: {len(df_model):,} diabetics with complete data")
print(f"Complication rate: {df_model['any_complication'].mean():.1%}")

# Check if we have enough cases
if len(df_model) < 100:
    print("\nWARNING: Small sample size may limit model performance")

# Prepare features and outcome
X = df_model[features]
y = df_model['any_complication']
weights = df_model['_LLCPWT']

print(f"\nFeature matrix shape: {X.shape}")
print(f"Complication cases: {y.sum():.0f}")
print(f"No complications: {(y==0).sum():.0f}")

# Feature summary
print("\nFeature Summary Statistics:")
print(X.describe().round(2))

# Check for class imbalance
complication_rate = y.mean()
print(f"\nClass balance:")
print(f"  No complication: {(1-complication_rate)*100:.1f}%")
print(f"  Complication: {complication_rate*100:.1f}%")

# Only proceed with modeling if we have sufficient cases
if y.sum() >= 10 and len(df_model) >= 50:
    
    # Train-test split
    X_train, X_test, y_train, y_test, w_train, w_test = train_test_split(
        X, y, weights,
        test_size=0.2,
        random_state=42,
        stratify=y if y.sum() >= 10 else None
    )
    
    print(f"\nTrain set: {len(X_train):,} ({y_train.mean():.1%} complications)")
    print(f"Test set: {len(X_test):,} ({y_test.mean():.1%} complications)")
    
    # Standardize features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # Fit weighted logistic regression
    print("\nFitting complication prediction model...")
    
    model = LogisticRegression(
        max_iter=1000,
        random_state=42,
        class_weight='balanced'  # Handle class imbalance
    )
    
    model.fit(X_train_scaled, y_train, sample_weight=w_train)
    
    print("Model trained successfully")
    
    # Model coefficients
    coef_df = pd.DataFrame({
        'feature': features,
        'coefficient': model.coef_[0],
        'odds_ratio': np.exp(model.coef_[0])
    })
    
    coef_df = coef_df.sort_values('odds_ratio', ascending=False)
    
    print("\n" + "="*70)
    print("COMPLICATION RISK FACTORS: ODDS RATIOS")
    print("="*70)
    print(coef_df.to_string(index=False))
    print("="*70)
    
    # Predictions
    y_pred_proba_train = model.predict_proba(X_train_scaled)[:, 1]
    y_pred_proba_test = model.predict_proba(X_test_scaled)[:, 1]
    
    # Calculate AUC
    try:
        auc_train = roc_auc_score(y_train, y_pred_proba_train, sample_weight=w_train)
        auc_test = roc_auc_score(y_test, y_pred_proba_test, sample_weight=w_test)
        
        print(f"\nModel Performance:")
        print(f"Training AUC: {auc_train:.3f}")
        print(f"Test AUC: {auc_test:.3f}")
    except:
        auc_train = np.nan
        auc_test = np.nan
        print("\nNote: Unable to calculate AUC (likely due to small sample size)")
    
    # Risk stratification
    # Assign risk scores to all diabetics with complete data
    df_model['risk_score'] = model.predict_proba(scaler.transform(X))[:, 1]
    
    # Create risk tiers
    df_model['risk_tier'] = pd.cut(
        df_model['risk_score'],
        bins=[0, 0.25, 0.50, 0.75, 1.0],
        labels=['Low', 'Medium', 'High', 'Very High']
    )
    
    print("\n" + "="*70)
    print("RISK STRATIFICATION")
    print("="*70)
    
    risk_summary = df_model.groupby('risk_tier').agg({
        'any_complication': ['count', 'sum', 'mean']
    }).round(3)
    
    risk_summary.columns = ['Total', 'Complications', 'Rate']
    print(risk_summary)
    print("="*70)
    
    # Create outputs
    os.makedirs('../outputs/figures', exist_ok=True)
    os.makedirs('../outputs/tables', exist_ok=True)
    
    # Figure 1: Risk Tier Distribution
    fig, ax = plt.subplots(figsize=(10, 6))
    
    risk_counts = df_model['risk_tier'].value_counts().sort_index()
    colors = ['green', 'yellow', 'orange', 'red']
    
    ax.bar(range(len(risk_counts)), risk_counts.values, color=colors, alpha=0.7)
    ax.set_xticks(range(len(risk_counts)))
    ax.set_xticklabels(risk_counts.index, fontsize=11)
    ax.set_ylabel('Number of Diabetics', fontsize=12, fontweight='bold')
    ax.set_xlabel('Risk Tier', fontsize=12, fontweight='bold')
    ax.set_title('Complication Risk Distribution Among Diabetics', 
                 fontsize=14, fontweight='bold', pad=20)
    
    for i, v in enumerate(risk_counts.values):
        ax.text(i, v + 5, str(v), ha='center', va='bottom', fontweight='bold')
    
    plt.tight_layout()
    plt.savefig('../outputs/figures/risk_tier_distribution.png', dpi=300, bbox_inches='tight')
    print("\nSaved: risk_tier_distribution.png")
    plt.close()
    
    # Figure 2: Complication Rate by Risk Tier
    fig, ax = plt.subplots(figsize=(10, 6))
    
    rates = risk_summary['Rate'].values
    tiers = risk_summary.index
    
    ax.bar(range(len(rates)), rates, color=colors, alpha=0.7)
    ax.set_xticks(range(len(rates)))
    ax.set_xticklabels(tiers, fontsize=11)
    ax.set_ylabel('Complication Rate', fontsize=12, fontweight='bold')
    ax.set_xlabel('Risk Tier', fontsize=12, fontweight='bold')
    ax.set_title('Complication Rate by Risk Tier', fontsize=14, fontweight='bold', pad=20)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{x*100:.0f}%'))
    
    for i, v in enumerate(rates):
        ax.text(i, v + 0.01, f'{v*100:.1f}%', ha='center', va='bottom', fontweight='bold')
    
    plt.tight_layout()
    plt.savefig('../outputs/figures/complication_rate_by_tier.png', dpi=300, bbox_inches='tight')
    print("Saved: complication_rate_by_tier.png")
    plt.close()
    
    # Figure 3: Odds Ratios for Complication Risk
    fig, ax = plt.subplots(figsize=(10, 8))
    
    coef_sorted = coef_df.sort_values('odds_ratio')
    y_pos = np.arange(len(coef_sorted))
    
    colors_or = ['red' if x < 1 else 'steelblue' for x in coef_sorted['odds_ratio']]
    ax.barh(y_pos, coef_sorted['odds_ratio'], color=colors_or, alpha=0.7)
    
    ax.axvline(x=1, color='black', linestyle='--', linewidth=1.5, label='OR = 1')
    ax.set_yticks(y_pos)
    ax.set_yticklabels(coef_sorted['feature'])
    ax.set_xlabel('Odds Ratio', fontsize=12, fontweight='bold')
    ax.set_title('Complication Risk Factors: Odds Ratios', fontsize=14, fontweight='bold', pad=20)
    ax.legend()
    
    for i, or_val in enumerate(coef_sorted['odds_ratio']):
        ax.text(or_val + 0.05, i, f'{or_val:.2f}', va='center', fontsize=9)
    
    plt.tight_layout()
    plt.savefig('../outputs/figures/complication_odds_ratios.png', dpi=300, bbox_inches='tight')
    print("Saved: complication_odds_ratios.png")
    plt.close()
    
    # Save results
    coef_df.to_csv('../outputs/tables/complication_model_coefficients.csv', index=False)
    print("Saved: complication_model_coefficients.csv")
    
    risk_summary.to_csv('../outputs/tables/risk_stratification.csv')
    print("Saved: risk_stratification.csv")
    
    # Intervention priority calculation
    # High-risk diabetics = highest ROI for intervention
    very_high_risk = df_model[df_model['risk_tier'] == 'Very High']
    
    intervention_summary = {
        'total_diabetics': int(len(df_model)),
        'very_high_risk_count': int(len(very_high_risk)),
        'very_high_risk_percent': float(len(very_high_risk) / len(df_model)),
        'very_high_risk_complication_rate': float(very_high_risk['any_complication'].mean()),
        'low_risk_complication_rate': float(df_model[df_model['risk_tier'] == 'Low']['any_complication'].mean()),
        'risk_ratio': float(
            very_high_risk['any_complication'].mean() / 
            df_model[df_model['risk_tier'] == 'Low']['any_complication'].mean()
        ) if len(df_model[df_model['risk_tier'] == 'Low']) > 0 else np.nan
    }
    
    with open('../outputs/tables/intervention_targeting.json', 'w') as f:
        json.dump(intervention_summary, f, indent=2)
    
    print("Saved: intervention_targeting.json")
    
    # Final summary
    print("\n" + "="*70)
    print("NOTEBOOK 5 COMPLETE: COMPLICATION PREDICTION")
    print("="*70)
    print(f"Diabetics analyzed: {len(df_model):,}")
    print(f"Overall complication rate: {df_model['any_complication'].mean():.1%}")
    print(f"\nRisk Stratification:")
    print(f"  Very High Risk: {len(df_model[df_model['risk_tier']=='Very High']):,} diabetics")
    print(f"  Complication rate: {df_model[df_model['risk_tier']=='Very High']['any_complication'].mean():.1%}")
    print(f"\nIntervention Targeting:")
    print(f"  Top {intervention_summary['very_high_risk_percent']*100:.1f}% of diabetics")
    print(f"  Account for highest complication risk")
    print(f"\nFigures created: 3")
    print(f"  - risk_tier_distribution.png")
    print(f"  - complication_rate_by_tier.png")
    print(f"  - complication_odds_ratios.png")
    print(f"\nTables created: 3")
    print(f"  - complication_model_coefficients.csv")
    print(f"  - risk_stratification.csv")
    print(f"  - intervention_targeting.json")
    print("="*70)

else:
    print("\nINSUFFICIENT DATA: Not enough complication cases for modeling")
    print("This is expected with small synthetic datasets")
    print("\nWith real BRFSS data (40K+ diabetics):")
    print("  - Sufficient power for complication modeling")
    print("  - Clear risk stratification")
    print("  - Intervention targeting feasible")

/var/folders/5l/3j7qr5tj6yl6_b8d9rn0grzr0000gn/T/ipykernel_50155/1893016104.py:5: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


Environment loaded successfully
Loaded: 10,000 respondents

Diabetics subset: 1,231 individuals
Percentage of total: 12.3%

Complication prevalence among diabetics:
  Cardiovascular: 5.7%
  Stroke: 3.5%
  Kidney disease: 2.7%
  Any complication: 11.4%

Creating risk prediction features...
Features created successfully

Modeling dataset: 1,155 diabetics with complete data
Complication rate: 11.3%

Feature matrix shape: (1155, 7)
Complication cases: 130
No complications: 1025

Feature Summary Statistics:
       diagnosis_age  age_group_num  disease_duration_proxy      bmi  \
count        1155.00        1155.00                 1155.00  1155.00   
mean           50.66           3.51                    4.31    33.13   
std            19.26           1.68                    9.13    10.22   
min            18.00           1.00                    0.00    15.10   
25%            34.00           2.00                    0.00    24.22   
50%            51.00           3.00                    0.00 

posx and posy should be finite values
posx and posy should be finite values
posx and posy should be finite values



Saved: risk_tier_distribution.png
Saved: complication_rate_by_tier.png
Saved: complication_odds_ratios.png
Saved: complication_model_coefficients.csv
Saved: risk_stratification.csv
Saved: intervention_targeting.json

NOTEBOOK 5 COMPLETE: COMPLICATION PREDICTION
Diabetics analyzed: 1,155
Overall complication rate: 11.3%

Risk Stratification:
  Very High Risk: 0 diabetics
  Complication rate: nan%

Intervention Targeting:
  Top 0.0% of diabetics
  Account for highest complication risk

Figures created: 3
  - risk_tier_distribution.png
  - complication_rate_by_tier.png
  - complication_odds_ratios.png

Tables created: 3
  - complication_model_coefficients.csv
  - risk_stratification.csv
  - intervention_targeting.json
